# Colab Bootstrap - LegalRAG

Notebook này **chỉ dùng để provision Colab server** và kiểm tra môi trường. Logic công việc chạy bằng **Colab Terminal** với script `.py` / CLI (không viết logic trong notebook).

## Quy trình mỗi session
1. Góc phải trên → **Select Kernel → Colab → Auto Connect** (hoặc **New Colab Server** để chọn máy có GPU).
2. Chạy cell **GPU check** bên dưới - phải thấy `cuda available: True` mới nên build embedding bằng GPU.
3. Mở **Colab: Open Terminal** (`Ctrl+Shift+P`) và làm việc bằng CLI / script `.py`:
   ```bash
   cd /content
   git clone https://github.com/ngminhduc129/LegalRAG.git
   cd LegalRAG && pip install -e .
   ```
4. VM là máy mới mỗi session: `git pull` sau mỗi lần push từ local, và tải artifact kết quả về trước khi session hết.

## Ghi chú
- Notebook không chứa secret. Không nhúng API key / token vào cell.
- Dùng cho Colab free: tài nguyên không đảm bảo, GPU có thể không được cấp.
- Nếu Colab Terminal (experimental) lỗi, xem cell fallback ở cuối notebook.

In [ ]:
# GPU + environment check (runs on the Colab runtime)
!nvidia-smi
import platform
import torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Package versions to record in build_meta.json (reproducibility)
import importlib.metadata as metadata

for pkg in ("torch", "transformers", "sentence-transformers", "numpy"):
    try:
        print(f"{pkg}: {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg}: not installed")

In [ ]:
# OPTIONAL: mount Google Drive (only needed if data lives on Drive)
# from google.colab import drive
# drive.mount("/content/drive")

## Fallback nếu Colab Terminal lỗi

Vẫn provision server bằng notebook, nhưng chạy lệnh ngay trong cell với prefix `!`:

```bash
!nvidia-smi
!cd /content && git clone https://github.com/ngminhduc129/LegalRAG.git
!cd /content/LegalRAG && pip install -e . -q
```

Sau đó (thay `<CHUNKS>` bằng file chunks đã upload):

```bash
!cd /content/LegalRAG && legal-rag index dense <CHUNKS> -o /content/out/dense_index.pkl
```